# 05 - PyTorch Modeli

Bu notebook, temel mimariyi PyTorch ile yeniden kurar ve NumPy ile Scikit-learn sonuclariyla karsilastirir.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_utils import RANDOM_SEED, prepare_datasets, set_global_seed
from src.metrics import evaluate_classification
from src.numpy_models import NumpyMLPClassifier
from src.pytorch_model import predict_torch, train_torch_model
from src.sklearn_model import train_sklearn_baseline
from src.visualization import plot_confusion_matrix_figure, plot_training_history


In [ ]:
set_global_seed(RANDOM_SEED)
DATA_PATH = PROJECT_ROOT / 'data' / 'heart_failure_clinical_records_dataset.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
split = prepare_datasets(DATA_PATH, random_state=RANDOM_SEED)

torch_result = train_torch_model(
    split.X_train,
    split.y_train,
    split.X_val,
    split.y_val,
    input_dim=split.X_train.shape[1],
    hidden_dim=8,
    learning_rate=0.03,
    epochs=250,
    batch_size=32,
    seed=RANDOM_SEED,
)

torch_test_pred, torch_test_prob = predict_torch(torch_result.model, split.X_test)
torch_metrics = evaluate_classification(split.y_test, torch_test_pred, torch_test_prob)
torch_metrics

In [ ]:
plot_training_history(torch_result.history, FIGURE_DIR / 'pytorch_baseline', 'PyTorch Baseline')
plot_confusion_matrix_figure(split.y_test, torch_test_pred, FIGURE_DIR / 'pytorch_confusion_matrix.png', 'PyTorch Baseline Confusion Matrix')
print(torch_metrics['classification_report'])

In [ ]:
numpy_baseline = NumpyMLPClassifier([split.X_train.shape[1], 8, 1], learning_rate=0.05, epochs=400, seed=RANDOM_SEED)
numpy_baseline.fit(split.X_train, split.y_train, split.X_val, split.y_val)
numpy_metrics = evaluate_classification(split.y_test, numpy_baseline.predict(split.X_test), numpy_baseline.predict_proba(split.X_test))

sklearn_model = train_sklearn_baseline(split.X_train, split.y_train)
sklearn_metrics = evaluate_classification(split.y_test, sklearn_model.predict(split.X_test), sklearn_model.predict_proba(split.X_test)[:, 1])

comparison_df = pd.DataFrame([
    {'model': 'NumPy baseline', 'accuracy': numpy_metrics['accuracy'], 'precision': numpy_metrics['precision'], 'recall': numpy_metrics['recall'], 'f1_score': numpy_metrics['f1_score']},
    {'model': 'Scikit-learn baseline', 'accuracy': sklearn_metrics['accuracy'], 'precision': sklearn_metrics['precision'], 'recall': sklearn_metrics['recall'], 'f1_score': sklearn_metrics['f1_score']},
    {'model': 'PyTorch baseline', 'accuracy': torch_metrics['accuracy'], 'precision': torch_metrics['precision'], 'recall': torch_metrics['recall'], 'f1_score': torch_metrics['f1_score']},
])
comparison_df

Yorum: PyTorch uygulamasi, temel NumPy model ile ayni katman duzeni, tanh gizli aktivasyonu, sigmoid cikis ve SGD optimizasyonunu kullanir. Bu nedenle burada amac yalnizca performans arttirmak degil, laboratuvar modelinin baska bir frameworkte tutarli bicimde yeniden kurulabildigini gostermektir.